# Module 04 - Full RAG Pipeline

**Duration:** 60 minutes

This module wires all the pieces together into a working system.
By the end you will have a RAG pipeline running against the sample documents,
a Gradio GUI to interact with it, and an understanding of how the LLM parameters
affect the quality and style of the answers.

---


## 4.1 Setting up the RAGTool

The `RAGTool` class in `src/ragsst/ragtool.py` is the central object in this repo.
It wraps the vector store, the LLM connection, and all the retrieval logic.

Before running the cells below, make sure Ollama is running:

```bash
ollama serve
```


In [ ]:
from ragsst.ragtool import RAGTool
import ragsst.parameters as p

print('Available embedding models:', p.EMBEDDING_MODELS)
print('Available LLMs:', p.LLM_CHOICES)


In [ ]:
tool = RAGTool(
    model='llama3.2',
    data_path='../data/sample_docs',
    collection_name='workshop_docs',
)

# This ingests documents if the collection is empty, or loads it if it already exists
tool.setup_vec_store()

print(f'Collection: {tool.collection_name}')
print(f'Documents in collection: {tool.collection.count()}')


## 4.2 Retrieval

Let us look at what the retriever actually returns before we involve the LLM.
This is worth doing separately because **retrieval failures are often the root cause
of bad answers** — and they are much easier to diagnose when you look at retrieval
in isolation.

### What the retriever actually does

When you call `tool.get_relevant_text(query, nresults=3)`:

1. Your query is embedded with the same model that was used at indexing time
2. ChromaDB performs approximate nearest-neighbour search in the vector index
3. The top-k most similar chunk vectors are returned
4. Their source text is concatenated and returned as a single string

The key constraint: **you must use the same embedding model for both indexing and querying.**
If you indexed with `multi-qa-mpnet-base-cos-v1` and query with `all-MiniLM-L6-v2`,
you are comparing apples to oranges. The `RAGTool` enforces this — it uses the same
`embedding_model` for both phases.

### Similarity threshold vs. top-k

Two parameters control what gets returned:

- **`nresults` (top-k)**: always return exactly this many chunks, ranked by similarity
- **`sim_th` (similarity threshold)**: filter out chunks with similarity below this value

They interact: if you set `nresults=5` and `sim_th=0.5`, you might get fewer than 5
results if some retrieved chunks score below 0.5.

Setting `sim_th` too high → missed answers (the right chunk gets filtered out)
Setting `sim_th` too low → noisy context (irrelevant chunks get included)


In [ ]:
query = 'What services does the AI service center offer?'

# get_relevant_text returns plain text, concatenated
context = tool.get_relevant_text(query, nresults=3, sim_th=0.3)
print('Retrieved context:')
print('-' * 60)
print(context)


In [ ]:
# retrieve_with_metadata shows similarity scores and sources
context_with_meta = tool.retrieve_with_metadata(query, nresults=3, sim_th=0.3)
print(context_with_meta)


The similarity score tells you how confident the retriever is.
A score below 0.3 usually means the question is about something not in the documents.
That is when you want the system to say 'I don't know' rather than hallucinate.


## 4.3 Prompt construction

The prompt is how we communicate the retrieved context and the user's question
to the LLM. Prompt quality matters more than most people realise.

### Anatomy of the RAG prompt

`tool.get_context_prompt(query, context)` produces something like:

```
Use the following context to answer the question.
If the answer is not clearly stated in the context, say "I don't have that information."
Keep the answer concise and factual.

Context:
[chunk 1 text]
[chunk 2 text]
[chunk 3 text]

Question: What AI workshops does the service center offer?
Answer:
```

### Why "answer only from the context" matters

Without this instruction, models often blend parametric memory with retrieved context.
For a fact-grounded Q&A system, this is dangerous: the model might answer correctly
*but from a different source than the one you retrieved*, making it impossible to
verify or attribute the answer.

The grounding instruction keeps the model honest — if it answers, it must be
able to point to the context as the source.

### The "lost in the middle" problem

Research has shown that LLMs tend to under-utilise information in the *middle*
of long context windows. They perform best on information at the beginning and end.

Practical implication: when you concatenate retrieved chunks, consider putting
the *most relevant* chunk first (or last), not in the middle.
Re-ranking (Module 05) helps identify which chunk is most relevant.


In [ ]:
context = tool.get_relevant_text(query, nresults=3)
prompt = tool.get_context_prompt(query, context)

print('Full prompt sent to LLM:')
print('=' * 60)
print(prompt)
print('=' * 60)


Read the prompt carefully. You can see exactly what the model is working from.
If the context does not contain the answer, the model should not be able to answer correctly.
When it does anyway, that is a hallucination.


## 4.4 Generation

Now we add the LLM step.


In [ ]:
# Single-turn RAG query
answer = tool.rag_query(
    user_msg=query,
    sim_th=0.3,
    nresults=3,
    top_k=5,
    top_p=0.9,
    temp=0.3,
)

print(f'Question: {query}')
print(f'\nAnswer: {answer}')


In [ ]:
# Try a few different questions
questions = [
    'Who is Sherlock Holmes?',
    'What happens at the end of Die Hard?',
    'What is the capital of Australia?',  # not in the documents
]

for q in questions:
    answer = tool.rag_query(q, sim_th=0.3, nresults=3, top_k=5, top_p=0.9, temp=0.3)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print()


The last question is not in the documents. What does the model do?
The answer depends on the similarity threshold. With a high threshold,
the retriever returns nothing and the model is told so.
With a low threshold, it retrieves something weakly related and may hallucinate.


## 4.5 LLM parameters

Three parameters control the randomness of the generated text.

**Temperature** (0 to 2): Controls how "random" the token sampling is.
- Temperature 0: always pick the highest-probability next token (deterministic, repetitive)
- Temperature 1: sample proportionally to the probability distribution (natural)
- Temperature > 1: flatten the distribution, increase surprise and creativity

For fact-grounded RAG answers, **low temperature (0.1–0.3)** is usually best.
You want the model to stay close to what the context says, not improvise.

**top_k**: Only consider the top-k highest-probability tokens at each step.
Lower values are more conservative. Values of 20–50 are typical.

**top_p** (nucleus sampling): Only consider tokens that together account for p% of
probability mass. A common default is 0.9.

### How these interact

Temperature is the most important of the three. `top_k` and `top_p` are
guardrails that prevent the temperature from occasionally sampling very unlikely tokens.

For RAG: keep temperature low. The model already has the answer in front of it —
creativity is not helpful here.
For open-ended generation: raise temperature (0.7–1.0) and let the model breathe.

### The difference between temperature 0 and temperature 0.1

Temperature 0 is fully deterministic (same input always produces same output).
Temperature 0.1 has a tiny amount of randomness that can actually *help*:
it avoids degenerate repetition loops that occasionally appear at temperature 0.
Many practitioners use 0.1 as their "deterministic" setting for this reason.


In [ ]:
# Compare different temperature settings on the same question
q = 'What happened to Hans Gruber at the end of the film?'

for temp in [0.1, 0.5, 0.9]:
    answer = tool.rag_query(q, sim_th=0.3, nresults=3, top_k=10, top_p=0.9, temp=temp)
    print(f'temp={temp}: {answer[:200]}')
    print()


## 4.6 Prompt engineering

The system prompt and instruction phrasing significantly affect the answer quality
and style. A few techniques that work reliably for RAG:

### 1. Be explicit about what to do when the answer is unknown

Bad: *"Answer the following question."*
Better: *"Use the context to answer. If the answer is not in the context, say 'I don't have that information.' Do not make up answers."*

The second version prevents the model from hallucinating when the context doesn't help.

### 2. Ask for citations

*"After your answer, cite the source by writing 'Source: [source name]'."*

This works because the metadata (source filename) is visible in the context.
It also helps you verify answers during evaluation.

### 3. Control answer length

*"Answer in 1–3 sentences."* or *"Provide a detailed answer with bullet points."*

LLMs default to verbose answers. For a chatbot or search result snippet, short
is usually better.

### 4. Chain-of-thought for complex questions

*"Think step by step before giving your final answer."*

Useful for questions that require reasoning over multiple retrieved chunks.
The model lays out its reasoning, which reduces errors and makes debugging easier.

### The prompt is just another hyperparameter

The prompt is not magic — it is an input to the model that you can iterate on.
Module 06 shows how to measure whether a prompt change actually improved things.
Don't tune prompts by feel; measure them.


In [ ]:
import requests, json
from os import getenv
from urllib.parse import urljoin

OLLAMA_URL = urljoin(getenv('OLLAMA_HOST', 'http://localhost:11434'), 'api')


def generate_with_prompt(prompt: str, temp: float = 0.3) -> str:
    r = requests.post(
        OLLAMA_URL + '/generate',
        json={'model': 'llama3.2', 'prompt': prompt, 'stream': False,
              'options': {'temperature': temp}}
    )
    return json.loads(r.text).get('response', '')


context = tool.get_relevant_text('What is the AI service center?', nresults=2)

prompt_v1 = f'Context:\n{context}\n\nQuestion: What is the AI service center?\nAnswer:'

prompt_v2 = (
    'You are a helpful assistant. Use only the context below to answer. '
    'If the answer is not in the context, say so.\n\n'
    f'Context:\n{context}\n\n'
    'Question: What is the AI service center?\n'
    'Answer in one sentence:'
)

print('Prompt v1 answer:')
print(generate_with_prompt(prompt_v1))
print('\nPrompt v2 answer:')
print(generate_with_prompt(prompt_v2))


## 4.7 The Gradio GUI

The repo includes a full web interface. Run the cell below to launch it.
You can also run it from the terminal: `uv run python local-rag-gui.py`


In [ ]:
from ragsst.interface import make_interface

gui = make_interface(tool)
gui.launch()


---

**Exercises**

1. Ask a question that requires information from two different documents
   (e.g. 'What do Die Hard and Sherlock Holmes have in common?').
   Set nresults=4. Does the answer draw from both?

2. Set sim_th=0.8 and ask a question. What changes? Lower it to 0.1.
   Where does the threshold need to be to get useful answers on your questions?

3. Write a system prompt that tells the model to always answer in Spanish.
   Does it comply? What happens when the context is in English?

---

**Further reading**

- Ollama API reference: https://github.com/ollama/ollama/blob/main/docs/api.md
- Gradio docs: https://www.gradio.app/docs
- Prompt engineering guide: https://www.promptingguide.ai/
